In [89]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import KBinsDiscretizer

In [91]:
df = pd.read_csv('./Dataset/train.csv', usecols = ['Age','Fare','Survived'])

In [93]:
df.head(5)

,Survived,Age,Fare
0,0,22.0,7.2500
1,1,38.0,71.2833
2,1,26.0,7.9250
3,1,35.0,53.1000
4,0,35.0,8.0500


In [95]:
df.dropna(inplace=True)

In [97]:
df.isnull().sum()

Survived    0
Age         0
Fare        0
dtype: int64

In [99]:
X_train,X_test,y_train,y_test = train_test_split(df.drop('Survived', axis = 1), 
                                                 df['Survived'], test_size = 0.3,
                                                 random_state = 42)

In [101]:
clf = DecisionTreeClassifier()

In [103]:
clf.fit(X_train,y_train)
y_pred = clf.predict(X_test)

In [105]:
accuracy_score(y_test,y_pred)

0.6

In [121]:
kbin_age = KBinsDiscretizer(n_bins = 20, encode = 'ordinal', strategy = 'kmeans')
kbin_fare = KBinsDiscretizer(n_bins = 20, encode = 'ordinal', strategy = 'kmeans')
# kbin_age = KBinsDiscretizer(n_bins = 5, encode = 'ordinal', strategy = 'kmeans') replace 'Kemans' with 'quantile'
# kbin_fare = KBinsDiscretizer(n_bins = 5, encode = 'ordinal', strategy = 'kmeans')

In [123]:
trf = ColumnTransformer([
    ('first', kbin_age,[0]),
    ('second', kbin_fare,[1])
])

In [125]:
X_train_trf = trf.fit_transform(X_train)
X_test_trf = trf.transform(X_test)

D:\Anaconda\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(
D:\Anaconda\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(


In [127]:
trf.named_transformers_['first'].bin_edges_

array([array([ 0.67      ,  4.37361111,  8.27777778, 12.5625    , 16.87372449,
              20.59693878, 24.53571429, 28.48728814, 32.59443099, 36.78683036,
              40.6333912 , 44.40013228, 48.35551948, 52.45454545, 56.68434343,
              60.38888889, 63.41666667, 67.60416667, 72.1875    , 77.        ,
              80.        ])                                                   ],
      dtype=object)

In [129]:
trf.named_transformers_['second'].bin_edges_

array([array([  0.        ,   4.52845576,  13.00639505,  22.11811429,
               32.46884881,  44.23405013,  53.62978346,  61.07327078,
               71.43096818,  80.85204026,  87.71889821, 100.93645625,
              115.48749375, 127.50499   , 143.33058286, 158.25894286,
              188.1224125 , 219.4515625 , 237.5229    , 255.1979    ,
              263.        ])                                         ],
      dtype=object)

In [130]:
clf = DecisionTreeClassifier()
clf.fit(X_train_trf, y_train)
y_pred2 = clf.predict(X_test_trf)

In [133]:
accuracy_score(y_test,y_pred2)

0.6325581395348837

In [135]:
output = pd.DataFrame({
    'age':X_train['Age'],
    'age_trf':X_train_trf[:,0],
    'fare':X_train['Fare'],
    'fare_trf':X_train_trf[:,1]
})

In [137]:
output['age_labels'] = pd.cut(x=X_train['Age'],
                                    bins=trf.named_transformers_['first'].bin_edges_[0].tolist())
output['fare_labels'] = pd.cut(x=X_train['Fare'],
                                    bins=trf.named_transformers_['second'].bin_edges_[0].tolist())

In [139]:
output.sample(5)

,age,age_trf,fare,fare_trf,age_labels,fare_labels
555,62.0,15.0,26.5500,3.0,"(60.389, 63.417]","(22.118, 32.469]"
780,13.0,3.0,7.2292,1.0,"(12.562, 16.874]","(4.528, 13.006]"
467,56.0,13.0,26.5500,3.0,"(52.455, 56.684]","(22.118, 32.469]"
246,25.0,6.0,7.7750,1.0,"(24.536, 28.487]","(4.528, 13.006]"
14,14.0,3.0,7.8542,1.0,"(12.562, 16.874]","(4.528, 13.006]"


In [145]:
def discretize(bins,strategy):
    kbin_age = KBinsDiscretizer(n_bins=bins,encode='ordinal',strategy=strategy)
    kbin_fare = KBinsDiscretizer(n_bins=bins,encode='ordinal',strategy=strategy)
    
    trf = ColumnTransformer([
        ('first',kbin_age,[0]),
        ('second',kbin_fare,[1])
    ])
    
    X_trf = trf.fit_transform(X)
    print(np.mean(cross_val_score(DecisionTreeClassifier(),df.drop('Survived', axis=1),df['Survived'],cv=10,scoring='accuracy')))
    
    plt.figure(figsize=(14,4))
    plt.subplot(121)
    plt.hist(X['Age'])
    plt.title("Before")

    plt.subplot(122)
    plt.hist(X_trf[:,0],color='red')
    plt.title("After")

    plt.show()
    
    plt.figure(figsize=(14,4))
    plt.subplot(121)
    plt.hist(X['Fare'])
    plt.title("Before")

    plt.subplot(122)
    plt.hist(X_trf[:,1],color='red')
    plt.title("Fare")

    plt.show()

In [147]:

discretize(5,'kmeans')

NameError: name 'X' is not defined